# STAR-v3 — Kaggle V4: TRAIN → INFERENCE tron goi (pose OFF · 500 query + ~13.7K gallery)

**Pipeline inference** (theo `inference2.svg`, da luoc theo quyet dinh):
`encode GLOBAL` → `Stage-1 cosine` → `Top-100` → **③ ITM cross-encoder re-rank** (BLIP-style: logit+cosine) → **⑤ Gale-Shapley** rank-1 → top-10/query.
**Bo:** ① Sinkhorn/DBSN · ④ ensemble · GNN/k-reciprocal (optional) · **ViTPose (tat han — train lan infer)**.

**Cai dat:** GPU T4 · Internet ON · **Add Input 2 dataset:**
1. `star-10k-hard` (nhu cu: `train_10k_hard_data.tar.zst` + `xvlm_16m_base.th`)
2. `star-oldtest` (`attr.json` + folder `test/` 1,978 jpg)

**Test set tu tao (merge):** 500 query old-test (seed 42, caption kieu nguoi viet) + gallery **13,713 anh**
= 1,978 real (500 GT + 1,478 distractor real) + 11,735 synthetic distractor (~thiet ke "500 + 15K").
⚠️ Tuyet doi KHONG dung gallery test (masked) cua nam nay — luat cam.

**Thoi gian uoc tinh:** setup ~10' + train ~60–85' + encode 13.7K ~8–10' + rerank 500×100 ~2–4' (+pure-1978 ~5') → **tong ~1.5–2h**.

In [ ]:
# [1/9] GPU + clone/pull repo (can ban MOI co src/star/inference)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
assert pathlib.Path("src/star/inference/pipeline.py").exists(), "repo cu — chua co module inference, pull lai!"
print("repo OK (co inference):", os.getcwd())

In [ ]:
# [2/9] Env pinned + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/9] Tim inputs: data 10K (tar.zst) + old-test (attr.json + test/) + X-VLM base
import glob, os, pathlib, shutil

def find_all(pattern):
    return sorted(set(glob.glob(f"/kaggle/input/*/{pattern}") +
                      glob.glob(f"/kaggle/input/*/**/{pattern}", recursive=True)))
def find_one(pattern):
    h = find_all(pattern)
    return h[0] if h else None

# --- data train 10K (can cho TRAIN va lam synthetic distractor) ---
ARCH = find_one("train_10k_hard_data.tar.zst")
marker = "/kaggle/working/extract"
got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
if not got:
    assert ARCH, "Khong thay train_10k_hard_data.tar.zst — Add Input dataset star-10k-hard!"
    os.makedirs(marker, exist_ok=True)
    print("extracting 1.6GB (~2-4 phut)...")
    if shutil.which("zstd"):
        os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
    else:
        import zstandard, tarfile
        with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
            with tarfile.open(fileobj=zr, mode="r|") as tf:
                tf.extractall(marker)
    got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
assert got, "giai nen that bai"
DATA_ROOT = str(pathlib.Path(got[0]).parents[2])

# --- old-test (1,978 anh that + GT) ---
attr = find_all("attr.json")
assert attr, "Khong thay attr.json — Add Input dataset star-oldtest!"
ATTR = attr[0]
OLD_ROOT = str(pathlib.Path(ATTR).parent)
probe = find_all("test/0.jpg") or find_all("0.jpg")
assert probe, "Khong thay anh test/0.jpg trong dataset old-test"
if not pathlib.Path(OLD_ROOT, "test/0.jpg").exists():
    OLD_ROOT = str(pathlib.Path(probe[0]).parents[1])

# --- X-VLM base ---
CKPT = find_one("xvlm_16m_base.th")
if not CKPT:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(CKPT).stat().st_size > 8e8
print("DATA_ROOT =", DATA_ROOT)
print("OLD_ROOT  =", OLD_ROOT)
print("CKPT      =", CKPT)

In [ ]:
# [4/9] Manifest TRAIN (pose OFF → KHONG can ViTPose; giu pair_image_id de bat group_by=pair neu muon)
import json, pathlib
import numpy as np, pandas as pd

root = pathlib.Path(DATA_ROOT)
anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")

for line in open(root / "data/subsets/train_10k_hard.jsonl", encoding="utf-8"):
    r = json.loads(line)
    anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"],
        pair_image_id=r.get("hard_i_id")))
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid, pair_image_id=None)

df = pd.DataFrame(anchors + [v for k, v in hard_rows.items() if k not in anchor_ids])

rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""

MANIFEST = "/kaggle/working/manifest_10k_hard.parquet"
df.to_parquet(MANIFEST, index=False)
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
print(f"rows={len(df)} train={(df.split=='train').sum()} "
      f"valb_q={((df.split=='valb') & (df.caption!='')).sum()} valb_d={((df.split=='valb') & (df.caption=='')).sum()} leakage={leak}")
assert leak == 0

In [ ]:
# [5/9] TRAIN (~60-85'): cau hinh tot nhat da biet (nen v2a) + pose OFF de khop inference khong-ViTPose
import pathlib, time, torch
DATA = f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT}"
TRAIN_SET = ("data.lhp_enabled=false model.pose_enabled=false "
             "loss.lambda_smooth_ap=0.2 optim.lr_lora=2e-4 "
             "optim.epochs=6 train.early_stop_patience=6 train.eval_every_epochs=0.5 "
             "train.batch_size=20 train.grad_accum=2 train.out_dir=/kaggle/working/v4")
# Neu V3 cho cai tien nao THANG RO (>+0.01 mAP), dan them flag vao TRAIN_SET:
#   " model.lora_freeze_text=false"                                      (v3a: mo LoRA text)
#   " model.lora_freeze_text=false model.lora_r=32 model.lora_alpha=64"  (v3b)
#   " data.group_by=pair optim.epochs=4"                                 (v3c: pair-batch)
cmd = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --set {DATA} {TRAIN_SET}"
print(cmd); t0 = time.time()
!{cmd}
mins = round((time.time() - t0) / 60, 1)
BEST = "/kaggle/working/v4/best.pth"
assert pathlib.Path(BEST).exists(), "train fail — xem log tren (neu OOM: ha train.batch_size=16)"
raw = torch.load(BEST, map_location="cpu", weights_only=False)
print(f"TRAIN xong ({mins} phut): VAL-B mAP = {raw.get('best_metric'):.4f} "
      f"| report: {(raw.get('extra') or {}).get('report')}")
del raw

In [ ]:
# [6/9] TEST SET MERGE: 500 query old-test (seed 42) + gallery 13,713 anh
#   = 1,978 real (500 GT + 1,478 distractor real) + 11,735 synthetic distractor
import glob, json, os
import numpy as np, pandas as pd

QUERY_MODE = "human"   # "human" = source_caption kieu nguoi viet (giong de nam nay) | "vlm" = caption chi tiet

rows = [json.loads(l) for l in open(ATTR, encoding="utf-8")]
assert len(rows) == 1978, f"attr.json co {len(rows)} dong (mong doi 1978)"
rng = np.random.default_rng(42)
pick = rng.permutation(len(rows))

def cap_of(r):
    if QUERY_MODE == "human":
        return (r.get("source_caption") or r["caption"])[0]
    return r["caption"][0]

out = []
for i in pick[:500]:                     # 500 QUERY — anh cua chinh no la GT trong gallery
    r = rows[i]
    out.append(dict(image_path=r["image"], caption=cap_of(r), split="valb",
                    sequence_id=f'old{r["image_id"]}', scene=f'old{r["image_id"]}',
                    action="query", image_id=f'old{r["image_id"]}', bbox=None, keypoints=None))
for i in pick[500:]:                     # 1,478 distractor REAL (kho nhat — cung domain voi GT)
    r = rows[i]
    out.append(dict(image_path=r["image"], caption="", split="valb",
                    sequence_id=f'old{r["image_id"]}', scene=f'old{r["image_id"]}',
                    action="distractor_real", image_id=f'old{r["image_id"]}', bbox=None, keypoints=None))
webp = sorted(glob.glob("/kaggle/working/extract/**/train_webp/**/*.webp", recursive=True))
assert len(webp) > 10000, f"chi thay {len(webp)} anh synthetic — kiem tra giai nen o cell 3"
for j, p in enumerate(webp):             # 11,735 distractor SYNTHETIC (path tuyet doi — dataset ho tro)
    out.append(dict(image_path=p, caption="", split="valb", sequence_id=f"syn{j}",
                    scene=f"syn{j}", action="distractor_syn", image_id=f"syn{j}",
                    bbox=None, keypoints=None))

test_df = pd.DataFrame(out)
TEST_MANIFEST = "/kaggle/working/test_v4_merged.parquet"
test_df.to_parquet(TEST_MANIFEST, index=False)

# --- KIEM TRA DU LIEU (phai xanh het) ---
assert test_df.image_id.nunique() == len(test_df), "image_id bi trung!"
assert (test_df.caption != "").sum() == 500, "so query khac 500!"
sample = [os.path.join(OLD_ROOT, rows[i]["image"]) for i in list(pick[:10]) + list(pick[-10:])]
sample += webp[:10] + webp[-10:]
missing = [p for p in sample if not os.path.exists(p)]
assert not missing, f"thieu file: {missing[:3]}"
print(f"TEST SET OK: {len(test_df)} rows = 500 query + 1,478 real-distractor + {len(webp)} synthetic-distractor")
print(f"gallery (dedup image_id) = {test_df.image_id.nunique()} anh")
print("vi du query:", test_df.caption.iloc[0][:100], "...")

In [ ]:
# [7/9] INFERENCE: Stage-1 cosine → Top-100 → ITM re-rank → Gale-Shapley  (~12-18')
RUN_PURE_1978 = True   # chay them gallery CHI 1,978 real (cung 500 query) de tach rieng tac dong distractor

cmd = (f"python scripts/run_inference.py --ckpt {BEST} --manifest {TEST_MANIFEST} "
       f"--image-root {OLD_ROOT} --base-ckpt {CKPT} --out-dir /kaggle/working/infer_v4 --topk 100")
print(cmd)
!{cmd}

if RUN_PURE_1978:
    import pandas as pd
    pure = pd.read_parquet(TEST_MANIFEST)
    pure = pure[~pure.image_id.str.startswith("syn")].reset_index(drop=True)
    PURE = "/kaggle/working/test_v4_pure1978.parquet"
    pure.to_parquet(PURE, index=False)
    cmd2 = (f"python scripts/run_inference.py --ckpt {BEST} --manifest {PURE} "
            f"--image-root {OLD_ROOT} --base-ckpt {CKPT} --out-dir /kaggle/working/infer_v4_pure --topk 100")
    print(cmd2)
    !{cmd2}

In [ ]:
# [8/9] BANG KET QUA 3 TANG (stage1 = cosine · rerank = +ITM · gale_shapley = +GS)
import json, pathlib
import pandas as pd
def load(p):
    f = pathlib.Path(p, "metrics.json")
    return json.loads(f.read_text()) if f.exists() else None
runs = {"merged_13.7K": load("/kaggle/working/infer_v4"),
        "pure_1978":    load("/kaggle/working/infer_v4_pure")}
tab_rows = []
for name, m in runs.items():
    if not m:
        continue
    for stage in ("stage1", "rerank", "gale_shapley"):
        tab_rows.append(dict(run=name, gallery=m["gallery_size"], stage=stage,
                             **{k: round(v, 4) for k, v in m[stage].items()}))
tab = pd.DataFrame(tab_rows)
print(tab.to_string(index=False))
print()
print("5 dong dau answer.txt (qi id1..id10):")
print("\n".join(pathlib.Path("/kaggle/working/infer_v4/answer.txt").read_text().splitlines()[:5]))
!ls -lh /kaggle/working/v4/best.pth /kaggle/working/infer_v4/ /kaggle/working/infer_v4_pure/ 2>/dev/null

## Doc ket qua
- **`rerank` − `stage1`** = dong gop cua cross-encoder ITM. Day la don chinh: tren VAL-B, R@1 0.53
  nhung R@5 0.83 → ~30% query co GT o rank 2–5, dung tam voi cua re-rank top-100.
- **`gale_shapley` − `rerank`** = dong gop cua rang buoc 1-1 (thuong nho; co the AM neu gia dinh
  "moi anh chi khop 1 query" khong dung — neu am thi khi infer that them `--no-gale-shapley`).
- **`merged_13.7K` vs `pure_1978`** = tac dong cua 11,735 distractor synthetic. Luu y 2 chieu:
  khac domain (de loai) NHUNG model DA THAY chung khi train (de bi hut nham) — so chenh do nhieu cross-domain thuc te.
- Chi 500 query → nhieu ~±0.02; doc XU HUONG, dung doc tung 0.005.

## Files can giu (Save Version → Output)
`v4/best.pth` · `infer_v4/metrics.json` · `infer_v4/answer.txt` · `infer_v4/ranks.pt` (+ ban `_pure`).

## Nhac luat
Test set (masked) nam nay **KHONG** duoc dung lam distractor/validation duoi moi hinh thuc.